# ENABL3S: independent methodological audit

Nine non-pilot healthy participants, six eligible under fixed rules, three excluded. This notebook reviews **saved derived results** and regenerates their report. It does not recompute original EMG amplitudes. Full raw reproduction requires the separate downloader and analysis (about 5.9 GB).

Source: Hu, Rouse & Hargrove, [ENABL3S v2](https://doi.org/10.6084/m9.figshare.5362627.v2), CC BY 4.0. Original project code/prose: MIT. Derived ENABL3S artifacts: CC BY 4.0; see DATA_SOURCES.md. No clinical validity or scientific-priority claim is made.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys
from IPython.display import display, Markdown, Image
roots = [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').exists():
    roots += [p.parents[1] for p in Path('/kaggle/input').rglob('reports/enabl3s_cohort_results_v1.json')]
root = next((p for p in roots if (p/'reports/enabl3s_cohort_results_v1.json').exists() and (p/'scripts/report_enabl3s_cohort.py').exists()), None)
if root is None:
    raise RuntimeError('Extract and attach the full source-and-results bundle; the notebook alone is insufficient.')
print('Python:', sys.version)
results_path = root/'reports/enabl3s_cohort_results_v1.json'
review = json.loads((root/'reports/enabl3s_cohort_review_verification_v1.json').read_text(encoding='utf-8'))
assert hashlib.sha256(results_path.read_bytes()).hexdigest() == review['results_sha256']
freeze = json.loads((root/'reports/enabl3s_cohort_protocol_freeze_v1.json').read_text(encoding='utf-8'))
for name, expected in freeze['files'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
data = json.loads(results_path.read_text(encoding='utf-8'))
assert data['summary']['participants_expected'] == 9
assert data['summary']['complete_primary'] == 6
assert data['summary']['excluded'] == 3
assert data['summary']['primary_crossings_of_one'] == 2
print(data['summary'])
print('Derived-artifact hashes checked; original participant archives were not re-read.')

## Regenerate the descriptive report

On Kaggle, copy the attached bundle to the writable working directory before regenerating. The source bundle remains untouched. Shared selections with no common blocks remain undefined. Alternative margins use the same original segment.

In [ ]:
import shutil, tempfile
working = Path(tempfile.mkdtemp(prefix='enabl3s-review-'))/'source'
working.mkdir(parents=True)
package_manifest = json.loads((root/'delivery/enabl3s/manifest.json').read_text(encoding='utf-8'))
for name in package_manifest['files']:
    relative = Path(name)
    if relative.is_absolute() or '..' in relative.parts:
        raise RuntimeError('Unsafe packaged path')
    destination = working/relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(root/relative, destination)
subprocess.run([sys.executable, str(working/'scripts/report_enabl3s_cohort.py')], cwd=working, check=True)
display(Markdown((working/'docs/enabl3s_cohort_results_v1.md').read_text(encoding='utf-8')))
display(Image(filename=str(working/'reports/figures/enabl3s_cohort_primary_v1.png')))
print('Regenerated review:', working)


## Recompute from original measurements

From the repository root with dependencies installed, run `python scripts/fetch_enabl3s_cohort.py`, `python scripts/analyze_enabl3s_cohort.py`, then `python scripts/report_enabl3s_cohort.py`. The downloader fetches pinned v2 metadata if absent, verifies each ZIP and records SHA-256. The analysis verifies the frozen protocol before reading signals.

This is methodological transport to healthy walking, not clinical validation in ACL reconstruction. A crossing of ratio 1 may be arbitrarily small. The six eligible participants are a selected subset; blocks and methods are not independent people. The existing Dryad Kaggle release and this extension have separate execution records.